In [6]:
import numpy as np
from sklearn.datasets import load_iris

In [18]:
iris = load_iris()
X= iris.data.astype(float)
y = iris.target
num_classes = 3 

Y = np.zeros((y.shape[0], num_classes), dtype=float) 
Y[np.arange(y.shape[0]), y] = 1.0

In [19]:
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
X= (X - X_mean)/ X_std

num_samples = X.shape[0]
train_size = 120
X_train, Y_train = X[:train_size], Y[:train_size]
X_test, Y_test = X[train_size:], Y[train_size:]

In [28]:
def sigmoid(z):
    return 1.0/(1.0 + np.exp(-z))

def sigmoid_derivative(a):
    return a*(1.0 - a)

def softmax(z):
    z_shift = z - np.max(z, axis=1, keepdims = True)
    exp_z = np.exp(z_shift)
    return exp_z/np.sum(exp_z, axis=1, keepdims=True)

def cross_entropy_loss(Y_true, Y_pred):
    eps = 1e-12
    Y_pred_clipped = np.clip(Y_pred, eps, 1.0 - eps)
    return -np.mean(np.sum(Y_true * np.log(Y_pred_clipped), axis =1))

def accuracy(Y_true, Y_pred):
    true_labels =np.argmax(Y_true, axis = 1)
    pred_labels = np.argmax(Y_pred, axis=1)
    return np.mean(true_labels == pred_labels)


In [29]:
class NeuralNetwork:
    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate = 0.1, seed = 42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0.0, np.sqrt(2.0 / (input_dim + hidden_dim)), size = (input_dim , hidden_dim))
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = rng.normal(0.0, np.sqrt(2.0 / (hidden_dim + output_dim)), size = (hidden_dim, output_dim))
        self.b2 = np.zeros((1, output_dim))
        self.lr = learning_rate

    def forward(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = sigmoid(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = softmax(z2)
        cache = {"X" : X, "z1" : z1, "a1" : a1, "z2" : z2, "a2" : a2}
        return a2, cache

    def backward(self, Y_true, cache):
        X, a1, a2, = cache["X"], cache["a1"], cache["a2"]
        N= X.shape[0]
        delta2 = (a2 - Y_true) / N
        dw2 = a1.T @ delta2
        db2 = np.sum(delta2, axis = 0, keepdims = True)

        delta1 = (delta2 @ self.W2.T) * sigmoid_derivative(a1)
        dw1 = X.T @ delta1
        db1 = np.sum(delta1, axis = 0, keepdims = True)

        return dw1, db1, dw2, db2

    def update(self, dw1, db1, dw2, db2):
        self.W1 -= self.lr * dw1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dw2
        self.b2 -= self.lr * db2

    def train(self, X, Y, epochs = 500):
        for epoch in range(1, epochs + 1):
            Y_pred, cache = self.forward(X)
            loss = cross_entropy_loss(Y, Y_pred)
            acc = accuracy(Y, Y_pred)

            dw1, db1, dw2, db2 = self.backward(Y, cache)
            self.update(dw1, db1, dw2, db2)

            if epoch % 50 == 0 or epoch == 1:
                print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Train Acc: {acc*100:.2f}%")

    def predict(self, X):
        Y_pred, _ = self.forward(X)
        return Y_pred

nn = NeuralNetwork(input_dim = 4, hidden_dim = 8, output_dim = 3, learning_rate = 0.5)
nn.train(X_train, Y_train, epochs=500)
        

Epoch    1 | Loss: 1.0617 | Train Acc: 46.67%
Epoch   50 | Loss: 0.3535 | Train Acc: 85.00%
Epoch  100 | Loss: 0.2557 | Train Acc: 91.67%
Epoch  150 | Loss: 0.1928 | Train Acc: 96.67%
Epoch  200 | Loss: 0.1433 | Train Acc: 98.33%
Epoch  250 | Loss: 0.1095 | Train Acc: 98.33%
Epoch  300 | Loss: 0.0879 | Train Acc: 98.33%
Epoch  350 | Loss: 0.0736 | Train Acc: 99.17%
Epoch  400 | Loss: 0.0636 | Train Acc: 100.00%
Epoch  450 | Loss: 0.0563 | Train Acc: 100.00%
Epoch  500 | Loss: 0.0507 | Train Acc: 100.00%
